Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import trino
import pandas as pd

conn = trino.dbapi.connect(
    host="trino-tableau-prod.dp.navi-tech.in",
    port=443,
    user="databricks-user",
    catalog="awsdatacatalog",
    schema="default",
    http_scheme="https",
)

query = """
with converted_cx_ids as(
select date(date_parse(converted_date, '%m/%d/%Y')) as converted_date, customer_id from corp_analytics.google_sheets_hl_sales_px_audits_funnel_tracker_data
),
interaction_data as (
SELECT 
    call_id,
    crt_object_id,
    process_name,
    lead_name,
    system_disposition,
    hangup_details,
    customer_talk_time,
    user_name,
    customer_talk_time_seconds,
    reference_id,
    date(parse_datetime(partition_key, 'yyyy-MM-dd-HH')) as call_date,
    cci.converted_date,
    case when date(parse_datetime(partition_key, 'yyyy-MM-dd-HH')) >= current_date - interval '35' day and cci.converted_date is null then 'd_1_nonsales_conversations'
         when cci.converted_date >= current_date - interval '35' day  then 'd_1_sales_conversations'
         else null
    end as allocation_type,
    coalesce (cci.converted_date, date(parse_datetime(partition_key, 'yyyy-MM-dd-HH'))) as audit_base_date
FROM corp_analytics_non_sensitive_view.hl_ameyo_customer_level_call_details_tbl am
left join converted_cx_ids cci on am.reference_id = cci.customer_id
WHERE cast(customer_talk_time_seconds as integer) > 0
)
select 
audit_base_date ,
CASE WHEN day_of_week(audit_base_date) = 6 THEN date_add('day', 2, audit_base_date) ELSE date_add('day', 1, audit_base_date) END AS Audit_Date,
sum(case when allocation_type = 'd_1_sales_conversations'  then 1 else 0 end) as sale_conversations_count,
sum(case when allocation_type = 'd_1_nonsales_conversations' then 1 else 0 end) as nonsales_conversations_count,
count (DISTINCT case when allocation_type = 'd_1_sales_conversations' then reference_id end ) as sale_cx_id_count 
from interaction_data
group by 1,2
order by 1 desc
"""

df = pd.read_sql(query, conn)
display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1oUhJ-fqDBj1T96a0rr3YcIzBbNFOeNsffHy-vsYr1Tg/edit?gid=827981589#gid=827981589"
worksheet_name = "Connected Call Count"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)